# Course 9: Introduction to Git

This notebook records core concepts, shell & Git workflows, commit internals (`Commit`, `Tree`, `Blob`), history inspection, diffing, and file restoration/reversion patterns from the **Introduction to Git** course on DataCamp (instructed by George Boorman).

---
## Chapter 1: Introduction to Version Control

### 1.1 What Is Version Control & Why Git?
* **Version Control**: Processes and systems used to manage changes to files, source code, datasets, and directories over time.
* **What Should Be Version Controlled?**: Any asset that **changes over time** or **needs to be shared/collaborated on** (scripts, notebooks, configuration files, documentation, small reference datasets).
* **Core Capabilities**:
  * Track files across multiple states and snapshots.
  * Compare versions across different points in time to audit **what** changed, **who** changed it, and **when**.
  * Combine different versions of files cleanly.
  * Revert mistakes and restore previous working states without losing history.
* **Git**: An open-source, distributed, highly scalable version control system executed primarily through the **shell (terminal)**.

### 1.2 Essential Shell Commands & Creating Repositories
* **Shell Navigation**:
  * `pwd` — Print working directory (current folder path).
  * `ls` — List files and sub-directories in the current directory.
  * `cd <directory>` — Change directory.
  * `git --version` — Verify Git installation and version.
* **Git Repository (`repo`)**: A project directory containing tracked files, sub-directories, and a hidden `.git/` directory that stores all version history and metadata (**never edit `.git/` directly**).
* **Initializing a Repository**:
  * `git init <project-name>` — Creates a new directory and initializes an empty Git repo inside it (e.g., `git init mental-health-workspace`).
  * `git init` — Converts an existing project directory into a Git repository.
* **Nested Repositories Warning**: Never run `git init` inside an existing Git repository. Creating multiple `.git` directories causes ambiguity over which repository tracks changes.

### 1.3 The Three-Stage Git Workflow (`Working Directory` $\rightarrow$ `Staging Area` $\rightarrow$ `Repository`)
1. **Working Directory**: Edit and save files locally on disk.
2. **Staging Area (`git add`)**: Select which modified or newly created files belong in the next snapshot:
   * `git add report.md` — Stage a single file.
   * `git add .` — Stage all new and modified files in the current directory and sub-directories.
3. **Repository (`git commit`)**: Permanently record a snapshot of the staging area in `.git/`:
   * `git commit -m "Adding a README."` — Commit with a concise inline message (`-m` avoids opening an interactive terminal editor).
4. **Inspecting State (`git status`)**: Displays the active branch (`main`), staged changes ready to commit, unstaged modifications, and untracked files.

In [ ]:
import os
import shutil
import subprocess
import tempfile

def run_git(cmd, cwd):
    if shutil.which("git") is None:
        print(f"$ {' '.join(cmd)}")
        print("[Note: `git` binary not detected in this runtime; execute locally with Git installed.]")
        print("-" * 60)
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(f"$ {' '.join(cmd)}")
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    print("-" * 60)
    return result

def git_exec(cmd, cwd):
    if shutil.which("git") is not None:
        subprocess.run(cmd, cwd=cwd, check=True, capture_output=True)

with tempfile.TemporaryDirectory() as tmpdir:
    run_git(["git", "--version"], tmpdir)

    # 1. Initialize a new repository: mental-health-workspace
    run_git(["git", "init", "-b", "main", "mental-health-workspace"], tmpdir)
    repo_dir = os.path.join(tmpdir, "mental-health-workspace")
    os.makedirs(os.path.join(repo_dir, "data"), exist_ok=True)

    git_exec(["git", "config", "user.name", "Rep Loop"], repo_dir)
    git_exec(["git", "config", "user.email", "repl@datacamp.com"], repo_dir)

    # 2. Check status on empty repo
    run_git(["git", "status"], repo_dir)

    # 3. Create project files and sub-directory
    with open(os.path.join(repo_dir, "README.md"), "w") as f:
        f.write("# Mental Health in Tech Workspace\n")
    with open(os.path.join(repo_dir, "report.md"), "w") as f:
        f.write("# Mental Health in Tech Survey\nTODO: write executive summary.\n")
    with open(os.path.join(repo_dir, "data", "mental_health_survey.csv"), "w") as f:
        f.write("age,gender,family_history,treatment\n31,M,No,No\n")

    # 4. Stage a single file first, then inspect git status
    run_git(["git", "add", "README.md"], repo_dir)
    run_git(["git", "status"], repo_dir)

    # 5. Commit README.md, then stage & commit all remaining files
    run_git(["git", "commit", "-m", "Adding a README."], repo_dir)
    run_git(["git", "add", "."], repo_dir)
    run_git(["git", "commit", "-m", "Add initial report and survey data."], repo_dir)


### Chapter 1: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Git tracks snapshots via a two-step save mechanism (**stage** with `git add`, then **snapshot** with `git commit`), giving you granular control over what enters project history.
* Always verify repository state with `git status` before and after staging to avoid committing unintended temporary files or creating nested `.git` directories.

#### Actionable Workflow: Starting a Version-Controlled Project
1. Navigate to your workspace (`cd path/to/project`) and verify no parent `.git` folder exists.
2. Run `git init` (or `git init <folder-name>` for a new folder).
3. Create/edit files, preview untracked files with `git status`, stage changes with `git add <file>` or `git add .`, and record a clear snapshot via `git commit -m "<concise summary>"`.

---
## Chapter 2: Viewing the Version History

### 2.1 Git Commit Architecture (`Commit`, `Tree`, `Blob`) & `git log`
Every Git commit consists of three internal object types identified by a 40-character SHA-1 **Git hash** (e.g., `b22eb75a82a68b9c0f1c45b9f5a9b7abe281683a`):
1. **Commit**: Stores commit metadata—author name/email, timestamp, log message, parent commit pointer, and root tree hash.
2. **Tree**: Maps directory structures and filenames to blobs and sub-trees (similar to a dictionary).
3. **Blob (Binary Large Object)**: Compressed snapshot of an individual file's contents. If two files have identical content, they share the exact same hash, allowing Git to deduplicate storage and compare files instantaneously.

* **`git log`**: Displays commit history in reverse chronological order (newest to oldest). In interactive terminal pagers, press `Space` to scroll forward and `q` to quit.

### 2.2 Filtering History & Inspecting Specific Commits
* **Limit Commit Count**: `git log -3` (shows the 3 most recent commits).
* **Filter by File Path**: `git log report.md` or combined `git log -2 mental_health_survey.csv`.
* **Filter by Date Range**:
  * `git log --since='Apr 2 2024'`
  * `git log --since='Apr 2 2024' --until='Apr 11 2024'`
  * Supports natural language (`"2 weeks ago"`, `"3 months ago"`, `"yesterday"`) and ISO format (`"YYYY-MM-DD"`, recommended to prevent regional day/month ambiguity; avoid commas like `"15 Jul, 2024"`).
* **Inspect a Single Commit (`git show`)**: `git show c27fa856` (first 8–10 hash characters suffice) outputs the commit's metadata, log message, and exact line-by-line diff.

### 2.3 Comparing Versions with `git diff`
* **Unstaged vs. Last Commit**: `git diff` (all files) or `git diff report.md` (single file).
* **Staged vs. Last Commit**: `git diff --staged` or `git diff --staged report.md`.
* **Between Two Commits**:
  * Using hashes (older hash first, newer hash second): `git diff 35f4b4d 186398f`
  * Using relative `HEAD` references: `git diff HEAD~1 HEAD` (`HEAD` = latest commit, `HEAD~1` = parent commit, `HEAD~2` = two commits prior).

### 2.4 Restoring & Reverting Changes (`revert`, `checkout`, `restore`)
* **Revert an Entire Commit (`git revert`)**: Creates a **new** commit that undoes the changes from a target commit while preserving complete project history:
  * `git revert HEAD` — Opens text editor to confirm commit message (`nano`: `Ctrl + O`, `Enter`, `Ctrl + X`).
  * `git revert --no-edit HEAD` — Reverts and commits immediately without opening an editor.
  * `git revert -n HEAD` — Reverts changes into the staging area without creating a commit yet (`--no-commit`).
* **Revert a Single File to a Past Commit (`git checkout`)**:
  * `git checkout HEAD~1 report.md` — Restores `report.md` from `HEAD~1` into the working directory and staging area, ready to commit.
* **Unstage Files Without Discarding Edits (`git restore --staged`)**:
  * `git restore --staged summary_statistics.csv` — Removes a single file from the staging area while keeping local edits intact.
  * `git restore --staged .` — Unstages all currently staged files.

In [ ]:
import os
import shutil
import subprocess
import tempfile

def run_git(cmd, cwd):
    if shutil.which("git") is None:
        print(f"$ {' '.join(cmd)}")
        print("[Note: `git` binary not detected in this runtime; execute locally with Git installed.]")
        print("-" * 60)
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(f"$ {' '.join(cmd)}")
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    print("-" * 60)
    return result

def git_exec(cmd, cwd):
    if shutil.which("git") is not None:
        subprocess.run(cmd, cwd=cwd, check=True, capture_output=True)

with tempfile.TemporaryDirectory() as tmpdir:
    git_exec(["git", "init", "-b", "main"], tmpdir)
    git_exec(["git", "config", "user.name", "Rep Loop"], tmpdir)
    git_exec(["git", "config", "user.email", "repl@datacamp.com"], tmpdir)

    report_path = os.path.join(tmpdir, "report.md")
    csv_path = os.path.join(tmpdir, "mental_health_survey.csv")

    # Commit 1
    with open(report_path, "w") as f:
        f.write("# Mental Health in Tech Survey\nTODO: write executive summary.\nTODO: add references.\n")
    with open(csv_path, "w") as f:
        f.write("age,gender,treatment\n31,M,No\n")
    git_exec(["git", "add", "."], tmpdir)
    git_exec(["git", "commit", "-m", "First commit: create report and survey data"], tmpdir)

    # Commit 2
    with open(report_path, "w") as f:
        f.write("# Mental Health in Tech Survey\nTODO: add references.\nTODO: cite funding sources.\n")
    git_exec(["git", "add", "report.md"], tmpdir)
    git_exec(["git", "commit", "-m", "Update TODO list in report.md"], tmpdir)

    # Commit 3 (accidental bad data entry)
    with open(csv_path, "a") as f:
        f.write("F,56,Yes\n")
    git_exec(["git", "add", "mental_health_survey.csv"], tmpdir)
    git_exec(["git", "commit", "-m", "Adding 50th participant data (corrupt row)"], tmpdir)

    # 1. Inspect commit log and compare HEAD~1 vs HEAD
    run_git(["git", "log", "-2"], tmpdir)
    run_git(["git", "diff", "HEAD~1", "HEAD"], tmpdir)

    # 2. Revert the bad commit cleanly without opening editor
    run_git(["git", "revert", "--no-edit", "HEAD"], tmpdir)

    # 3. Demonstrate staging & unstaging with git restore --staged
    with open(report_path, "a") as f:
        f.write("Draft note to unstage.\n")
    run_git(["git", "add", "report.md"], tmpdir)
    run_git(["git", "diff", "--staged", "report.md"], tmpdir)
    run_git(["git", "restore", "--staged", "report.md"], tmpdir)
    run_git(["git", "status", "-s"], tmpdir)


### Chapter 2: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Git stores history as immutable snapshots of **Commits**, **Trees**, and **Blobs** linked by SHA-1 hashes.
* Use `git diff` (unstaged), `git diff --staged` (staged), and `git diff HEAD~1 HEAD` (committed) to audit exact line changes before committing or reverting.
* Prefer `git revert --no-edit <commit>` over destructive history rewrites when undoing committed mistakes so the audit trail remains intact.

#### Complete Command Reference — Introduction to Git
| Command | Purpose / Effect |
| :--- | :--- |
| `git init [dir]` | Initialize a new Git repository (or convert an existing directory) |
| `git status` | Show current branch, staged files, unstaged edits, and untracked files |
| `git add <file>` / `git add .` | Add a specific file or all modified/new files to the staging area |
| `git commit -m "<msg>"` | Record staged changes as a new commit snapshot with a message |
| `git log [-n] [--since] [--until]` | View commit history, optionally filtered by count, file, or date range |
| `git show <hash>` | Inspect commit metadata and diff for a specific commit hash |
| `git diff [file]` | Compare unstaged working directory changes against the latest commit |
| `git diff --staged [file]` | Compare staged changes against the latest commit |
| `git diff <hash1> <hash2>` | Compare changes between two commits (or `HEAD~1 HEAD`) |
| `git revert [--no-edit \| -n] <ref>` | Create a new commit (or stage changes with `-n`) undoing a prior commit |
| `git checkout <ref> <file>` | Restore a single file from a historical commit into the staging area |
| `git restore --staged <file\|.>` | Remove file(s) from the staging area without losing local modifications |